### assumes ./data is populated w/ ambignq dataset alr (if not run ./download_ambignq_dataset.ipynb)

In [ ]:
# Install required packages (run once)
!pip install -q pandas matplotlib seaborn tqdm scikit-learn

In [2]:
import json
import requests
import zipfile
from pathlib import Path
import shutil

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Setup complete!")

✅ Setup complete!


/opt/anaconda3/envs/182_proj/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
DATA_DIR = Path("./data")

In [6]:
def load_split(split_name):
    """
    Load one split of the data.
    
    Args:
        split_name: 'train' or 'dev'
    """
    file_path = DATA_DIR / f"{split_name}_light.json"
    
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    print(f"✅ Loaded {len(data):,} examples from {split_name}")
    return data

# Load available splits
train_data = load_split("train")
dev_data = load_split("dev")

print(f"\n📊 Total: {len(train_data) + len(dev_data):,} examples")

✅ Loaded 10,036 examples from train
✅ Loaded 2,002 examples from dev

📊 Total: 12,038 examples


##### note: i believe dev = validation set 
##### below we are going to construct a dataset of ambiguous + unambiguous question pairs

In [9]:
print(len(train_data))

10036


In [11]:
train_data[0]

{'annotations': [{'type': 'multipleQAs',
   'qaPairs': [{'question': 'When did the Simpsons first air on television as an animated short on the Tracey Ullman Show?',
     'answer': ['April 19, 1987']},
    {'question': 'When did the Simpsons first air as a half-hour prime time show?',
     'answer': ['December 17, 1989']}]}],
 'id': '-4469503464110108318',
 'question': 'When did the simpsons first air on television?'}

In [19]:
from collections import Counter

types = [train_data[i]["annotations"][0]["type"] for i in range(len(train_data))]
cnter = Counter(types)
cnter

Counter({'singleAnswer': 5289, 'multipleQAs': 4747})

In [20]:
ambig_datapoints = [train_data[i] for i in range(len(train_data)) if train_data[i]["annotations"][0]["type"] == "multipleQAs"]
len(ambig_datapoints)

4747

In [23]:
num_annotations = [len(ambig_datapoints[i]["annotations"][0]["qaPairs"]) for i in range(len(ambig_datapoints))]
len(num_annotations)

4747

In [24]:
import pandas as pd
num_annots_series = pd.Series(num_annotations)
num_annots_series.describe()

count    4747.000000
mean        2.939751
std         1.326158
min         2.000000
25%         2.000000
50%         3.000000
75%         3.000000
max        11.000000
dtype: float64

##### making 2 pairs per ambiguous question (as min annotations is 2) --> can tweak this code to make it 3 or 4 pairs in the future but i claim using the nonlight ambignq dataset might be more positive

In [35]:
num_pairs_per = 2    # can modify this (but some exampes only have 2 annots)

data = []

for annot in ambig_datapoints:
    unambig_qs = [annot["annotations"][0]["qaPairs"][i]["question"] for i in range(len(annot["annotations"][0]["qaPairs"]))]
    
    for i in range(min(num_pairs_per, len(unambig_qs))):
        dct = {}
        dct["id"] = annot["id"]
        dct["ambiguous_question"] = annot["question"]
        dct["unambiguous_question"] = unambig_qs[i]

        data.append(dct)

len(data)    

9494

In [37]:
data

[{'id': '-4469503464110108318',
  'ambiguous_question': 'When did the simpsons first air on television?',
  'unambiguous_question': 'When did the Simpsons first air on television as an animated short on the Tracey Ullman Show?'},
 {'id': '-4469503464110108318',
  'ambiguous_question': 'When did the simpsons first air on television?',
  'unambiguous_question': 'When did the Simpsons first air as a half-hour prime time show?'},
 {'id': '-6631915997977101143',
  'ambiguous_question': 'What is the legal age of marriage in usa?',
  'unambiguous_question': 'What is the legal age of marriage, without parental consent or other authorization, in all but two states in the usa?'},
 {'id': '-6631915997977101143',
  'ambiguous_question': 'What is the legal age of marriage in usa?',
  'unambiguous_question': 'What is the legal age of marriage, without parental consent or other authorization, in Nebraska?'},
 {'id': '-3098213414945179817',
  'ambiguous_question': 'Who starred in barefoot in the park 